# ydata-profiling + PII Classification → SQL Server

Extracts profiling results **and** probabilistic PII detection (via Microsoft Presidio) and stores them across 6 SQL Server tables:

| Table | Contents |
|---|---|
| `metadata_dataset_overview` | Row/col counts, duplicates, missing % |
| `metadata_column_stats` | Type, missing count/%, unique count/% |
| `metadata_descriptive_stats` | mean, std, min, max, quartiles, skewness, kurtosis |
| `metadata_correlations` | Pairwise Pearson correlations |
| `metadata_pii_column_summary` | Per-column PII entity types, avg/max confidence score |
| `metadata_pii_cell_detail` | Per-cell PII flags with entity type and confidence score |

## 1. Install dependencies

In [ ]:
# Run once if not already installed
# !pip install ydata-profiling pyodbc sqlalchemy pandas
# !pip install presidio-analyzer presidio-anonymizer
# !python -m spacy download en_core_web_lg   # recommended for best NER accuracy
# !python -m spacy download en_core_web_sm   # lighter alternative

## 2. Imports

In [ ]:
import pandas as pd
import numpy as np
from ydata_profiling import ProfileReport
from sqlalchemy import create_engine, text
from presidio_analyzer import AnalyzerEngine, RecognizerRegistry
from presidio_analyzer.nlp_engine import NlpEngineProvider
import urllib
from datetime import datetime
from collections import defaultdict

## 3. SQL Server connection
Choose **Option A** (Windows Auth) or **Option B** (Username & Password) and comment out the other.

In [ ]:
# ── CONFIGURE THESE ───────────────────────────────────────────────────────────
SQL_SERVER   = "YOUR_SERVER_NAME"      # e.g. 'localhost' or 'myserver.database.windows.net'
SQL_DATABASE = "YOUR_DATABASE_NAME"
SQL_SCHEMA   = "dbo"

# Option A: Windows Authentication (trusted connection) -----------------------
params = urllib.parse.quote_plus(
    f"DRIVER={{ODBC Driver 17 for SQL Server}};"
    f"SERVER={SQL_SERVER};DATABASE={SQL_DATABASE};"
    f"Trusted_Connection=yes;"
)

# Option B: Username & Password -----------------------------------------------
# SQL_USERNAME = "YOUR_USERNAME"
# SQL_PASSWORD = "YOUR_PASSWORD"
# params = urllib.parse.quote_plus(
#     f"DRIVER={{ODBC Driver 17 for SQL Server}};"
#     f"SERVER={SQL_SERVER};DATABASE={SQL_DATABASE};"
#     f"UID={SQL_USERNAME};PWD={SQL_PASSWORD};"
# )

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={params}")
print("Engine created. Testing connection...")
with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
print("Connection successful.")

## 4. Load your data

In [ ]:
# ── REPLACE WITH YOUR DATA ────────────────────────────────────────────────────
# df = pd.read_csv("your_file.csv")
# df = pd.read_excel("your_file.xlsx")
# df = pd.read_sql("SELECT * FROM your_table", engine)

# Sample data for demonstration
df = pd.DataFrame({
    "full_name":    ["Alice Smith", "Bob Jones", None, "Carol White", "Dave Brown"],
    "email":        ["alice@example.com", "bob@test.org", "carol@mail.com", None, "dave@company.com"],
    "phone":        ["+1-800-555-0100", "212-555-0199", None, "415-555-0123", "+1-212-555-0155"],
    "ssn":          ["123-45-6789", None, "987-65-4321", "111-22-3333", None],
    "credit_card":  ["4111111111111111", None, "5500005555555559", None, "340000000000009"],
    "dob":          ["1990-04-15", "1985-07-22", "2000-01-30", None, "1978-11-05"],
    "ip_address":   ["192.168.1.1", "10.0.0.1", None, "172.16.254.1", "192.168.0.100"],
    "salary":       [50000, 62000, 71000, None, 48000],
    "department":   ["HR", "Engineering", "Finance", "HR", "Engineering"],
})

DATASET_NAME = "sample_employees"
RUN_TS       = datetime.utcnow()

df.head()

## 5. Run ydata-profiling

In [ ]:
profile = ProfileReport(df, title=DATASET_NAME, explorative=True)

# Optional: view inline
# profile.to_notebook_iframe()

## 6. Extract ydata-profiling metadata

In [ ]:
desc        = profile.get_description()
table_stats = desc.table

# ── 6a. Dataset overview ──────────────────────────────────────────────────────
overview_df = pd.DataFrame([{
    "dataset_name":       DATASET_NAME,
    "run_timestamp":      RUN_TS,
    "n_rows":             int(table_stats.get("n", 0)),
    "n_cols":             int(table_stats.get("n_var", 0)),
    "n_missing_cells":    int(table_stats.get("n_cells_missing", 0)),
    "pct_missing_cells":  round(float(table_stats.get("p_cells_missing", 0)) * 100, 4),
    "n_duplicate_rows":   int(table_stats.get("n_duplicates", 0)),
    "pct_duplicate_rows": round(float(table_stats.get("p_duplicates", 0)) * 100, 4),
    "n_numeric_cols":     int(table_stats.get("types", {}).get("Numeric", 0)),
    "n_categorical_cols": int(table_stats.get("types", {}).get("Categorical", 0)),
}])

# ── 6b. Column stats + descriptive stats ──────────────────────────────────────
col_stats_rows, desc_stats_rows = [], []
n_rows = int(table_stats.get("n", 1))

for col_name, col_data in desc.variables.items():
    col_type = str(col_data.get("type", "Unknown"))

    col_stats_rows.append({
        "dataset_name":  DATASET_NAME,
        "run_timestamp": RUN_TS,
        "column_name":   col_name,
        "column_type":   col_type,
        "n_missing":     int(col_data.get("n_missing", 0)),
        "pct_missing":   round(float(col_data.get("p_missing", 0)) * 100, 4),
        "n_unique":      int(col_data.get("n_unique", col_data.get("n_distinct", 0))),
        "pct_unique":    round(int(col_data.get("n_unique", col_data.get("n_distinct", 0))) / n_rows * 100, 4),
    })

    if col_type in ("Numeric", "NUM") or col_data.get("mean") is not None:
        desc_stats_rows.append({
            "dataset_name":  DATASET_NAME,
            "run_timestamp": RUN_TS,
            "column_name":   col_name,
            "mean":          round(float(col_data.get("mean",     0) or 0), 6),
            "std":           round(float(col_data.get("std",      0) or 0), 6),
            "min":           round(float(col_data.get("min",      0) or 0), 6),
            "pct_25":        round(float(col_data.get("25%",      0) or 0), 6),
            "median":        round(float(col_data.get("50%",      0) or 0), 6),
            "pct_75":        round(float(col_data.get("75%",      0) or 0), 6),
            "max":           round(float(col_data.get("max",      0) or 0), 6),
            "kurtosis":      round(float(col_data.get("kurtosis", 0) or 0), 6),
            "skewness":      round(float(col_data.get("skewness", 0) or 0), 6),
        })

col_stats_df  = pd.DataFrame(col_stats_rows)
desc_stats_df = pd.DataFrame(desc_stats_rows)

# ── 6c. Correlations ──────────────────────────────────────────────────────────
corr_rows = []
if desc.correlations:
    corr_key = "pearson" if "pearson" in desc.correlations else next(iter(desc.correlations), None)
    if corr_key:
        corr_matrix = desc.correlations[corr_key]
        for col_a in corr_matrix.columns:
            for col_b in corr_matrix.columns:
                if col_a < col_b:
                    val = corr_matrix.loc[col_a, col_b] if col_a in corr_matrix.index else None
                    if val is not None and not pd.isna(val):
                        corr_rows.append({
                            "dataset_name":     DATASET_NAME,
                            "run_timestamp":    RUN_TS,
                            "correlation_type": corr_key,
                            "column_a":         col_a,
                            "column_b":         col_b,
                            "correlation_value":round(float(val), 6),
                        })

corr_df = pd.DataFrame(corr_rows)

print("ydata-profiling metadata extracted.")
display(overview_df)
display(col_stats_df)

## 7. PII detection with Microsoft Presidio

Scans every cell in every **string / object column** and produces:
- **Per-cell detail**: row index, column, entity type, confidence score
- **Per-column summary**: entity types detected, avg confidence, max confidence, % of rows flagged

In [ ]:
# ── PII entity types to detect ────────────────────────────────────────────────
PII_ENTITIES = [
    "PERSON",           # Names
    "EMAIL_ADDRESS",    # Email addresses
    "PHONE_NUMBER",     # Phone numbers
    "US_SSN",           # SSN / national IDs
    "CREDIT_CARD",      # Credit card numbers
    "DATE_TIME",        # Dates of birth (and other dates)
    "LOCATION",         # Addresses
    "IP_ADDRESS",       # IP addresses
]

# ── Confidence threshold — detections below this score are ignored ─────────────
CONFIDENCE_THRESHOLD = 0.4

# ── Initialise Presidio ───────────────────────────────────────────────────────
# Uses en_core_web_lg if available, falls back to en_core_web_sm
try:
    nlp_config = {"nlp_engine_name": "spacy", "models": [{"lang_code": "en", "model_name": "en_core_web_lg"}]}
    provider   = NlpEngineProvider(nlp_configuration=nlp_config)
    nlp_engine = provider.create_engine()
    analyzer   = AnalyzerEngine(nlp_engine=nlp_engine)
    print("Using en_core_web_lg")
except Exception:
    nlp_config = {"nlp_engine_name": "spacy", "models": [{"lang_code": "en", "model_name": "en_core_web_sm"}]}
    provider   = NlpEngineProvider(nlp_configuration=nlp_config)
    nlp_engine = provider.create_engine()
    analyzer   = AnalyzerEngine(nlp_engine=nlp_engine)
    print("Falling back to en_core_web_sm")

In [ ]:
# ── Scan every string column ──────────────────────────────────────────────────
pii_cell_rows   = []
pii_col_staging = defaultdict(list)   # col_name → list of (entity_type, score)

string_cols = df.select_dtypes(include=["object", "string"]).columns.tolist()
print(f"Scanning {len(string_cols)} string column(s): {string_cols}\n")

for col in string_cols:
    for row_idx, cell_value in df[col].items():
        if pd.isna(cell_value) or str(cell_value).strip() == "":
            continue

        results = analyzer.analyze(
            text=str(cell_value),
            entities=PII_ENTITIES,
            language="en"
        )

        for result in results:
            if result.score < CONFIDENCE_THRESHOLD:
                continue

            pii_cell_rows.append({
                "dataset_name":  DATASET_NAME,
                "run_timestamp": RUN_TS,
                "column_name":   col,
                "row_index":     int(row_idx),
                "cell_value":    str(cell_value),      # store original value
                "entity_type":   result.entity_type,
                "confidence":    round(result.score, 6),
                "start_char":    result.start,
                "end_char":      result.end,
            })

            pii_col_staging[col].append((result.entity_type, result.score))

pii_cell_df = pd.DataFrame(pii_cell_rows) if pii_cell_rows else pd.DataFrame(
    columns=["dataset_name","run_timestamp","column_name","row_index",
             "cell_value","entity_type","confidence","start_char","end_char"]
)

print(f"Cell-level PII detections: {len(pii_cell_df)}")
pii_cell_df.head(10)

In [ ]:
# ── Build per-column PII summary ──────────────────────────────────────────────
pii_col_rows = []

for col in string_cols:
    detections = pii_col_staging.get(col, [])
    n_non_null = int(df[col].notna().sum())

    if detections:
        entity_types   = list({e for e, _ in detections})   # unique entity types
        scores         = [s for _, s in detections]
        flagged_rows   = pii_cell_df[pii_cell_df["column_name"] == col]["row_index"].nunique()
        is_pii         = True
    else:
        entity_types   = []
        scores         = []
        flagged_rows   = 0
        is_pii         = False

    pii_col_rows.append({
        "dataset_name":         DATASET_NAME,
        "run_timestamp":        RUN_TS,
        "column_name":          col,
        "is_pii":               is_pii,
        "detected_entity_types":  ", ".join(sorted(entity_types)) if entity_types else None,
        "n_entity_types":       len(entity_types),
        "avg_confidence":       round(float(np.mean(scores)), 6)   if scores else None,
        "max_confidence":       round(float(np.max(scores)),  6)   if scores else None,
        "min_confidence":       round(float(np.min(scores)),  6)   if scores else None,
        "n_flagged_rows":        flagged_rows,
        "pct_flagged_rows":      round(flagged_rows / n_non_null * 100, 4) if n_non_null > 0 else 0.0,
        "confidence_threshold": CONFIDENCE_THRESHOLD,
    })

pii_col_df = pd.DataFrame(pii_col_rows)
print("Per-column PII summary:")
pii_col_df

## 8. Auto-create all SQL Server tables

In [ ]:
DDL_STATEMENTS = [

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'metadata_dataset_overview')
CREATE TABLE [{SQL_SCHEMA}].[metadata_dataset_overview] (
    id                   INT IDENTITY(1,1) PRIMARY KEY,
    dataset_name         NVARCHAR(255)  NOT NULL,
    run_timestamp        DATETIME2      NOT NULL,
    n_rows               INT,
    n_cols               INT,
    n_missing_cells      INT,
    pct_missing_cells    FLOAT,
    n_duplicate_rows     INT,
    pct_duplicate_rows   FLOAT,
    n_numeric_cols       INT,
    n_categorical_cols   INT
);""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'metadata_column_stats')
CREATE TABLE [{SQL_SCHEMA}].[metadata_column_stats] (
    id               INT IDENTITY(1,1) PRIMARY KEY,
    dataset_name     NVARCHAR(255)  NOT NULL,
    run_timestamp    DATETIME2      NOT NULL,
    column_name      NVARCHAR(255)  NOT NULL,
    column_type      NVARCHAR(100),
    n_missing        INT,
    pct_missing      FLOAT,
    n_unique         INT,
    pct_unique       FLOAT
);""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'metadata_descriptive_stats')
CREATE TABLE [{SQL_SCHEMA}].[metadata_descriptive_stats] (
    id               INT IDENTITY(1,1) PRIMARY KEY,
    dataset_name     NVARCHAR(255)  NOT NULL,
    run_timestamp    DATETIME2      NOT NULL,
    column_name      NVARCHAR(255)  NOT NULL,
    mean             FLOAT,
    std              FLOAT,
    min              FLOAT,
    pct_25           FLOAT,
    median           FLOAT,
    pct_75           FLOAT,
    max              FLOAT,
    kurtosis         FLOAT,
    skewness         FLOAT
);""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'metadata_correlations')
CREATE TABLE [{SQL_SCHEMA}].[metadata_correlations] (
    id                  INT IDENTITY(1,1) PRIMARY KEY,
    dataset_name        NVARCHAR(255)  NOT NULL,
    run_timestamp       DATETIME2      NOT NULL,
    correlation_type    NVARCHAR(50),
    column_a            NVARCHAR(255)  NOT NULL,
    column_b            NVARCHAR(255)  NOT NULL,
    correlation_value   FLOAT
);""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'metadata_pii_column_summary')
CREATE TABLE [{SQL_SCHEMA}].[metadata_pii_column_summary] (
    id                      INT IDENTITY(1,1) PRIMARY KEY,
    dataset_name            NVARCHAR(255)  NOT NULL,
    run_timestamp           DATETIME2      NOT NULL,
    column_name             NVARCHAR(255)  NOT NULL,
    is_pii                  BIT            NOT NULL,
    detected_entity_types   NVARCHAR(500),             -- comma-separated list
    n_entity_types          INT,
    avg_confidence          FLOAT,
    max_confidence          FLOAT,
    min_confidence          FLOAT,
    n_flagged_rows          INT,
    pct_flagged_rows        FLOAT,
    confidence_threshold    FLOAT
);""",

f"""IF NOT EXISTS (SELECT 1 FROM sys.tables WHERE name = 'metadata_pii_cell_detail')
CREATE TABLE [{SQL_SCHEMA}].[metadata_pii_cell_detail] (
    id               INT IDENTITY(1,1) PRIMARY KEY,
    dataset_name     NVARCHAR(255)  NOT NULL,
    run_timestamp    DATETIME2      NOT NULL,
    column_name      NVARCHAR(255)  NOT NULL,
    row_index        INT            NOT NULL,
    cell_value       NVARCHAR(MAX),
    entity_type      NVARCHAR(100)  NOT NULL,
    confidence       FLOAT          NOT NULL,
    start_char       INT,
    end_char         INT
);"""
]

with engine.begin() as conn:
    for stmt in DDL_STATEMENTS:
        conn.execute(text(stmt))

print("All 6 tables created (or already exist).")

## 9. Insert all metadata into SQL Server

In [ ]:
TABLE_MAP = {
    "metadata_dataset_overview":   overview_df,
    "metadata_column_stats":       col_stats_df,
    "metadata_descriptive_stats":  desc_stats_df,
    "metadata_correlations":       corr_df,
    "metadata_pii_column_summary": pii_col_df,
    "metadata_pii_cell_detail":    pii_cell_df,
}

with engine.begin() as conn:
    for table_name, df_to_insert in TABLE_MAP.items():
        if df_to_insert.empty:
            print(f"  Skipped {table_name} (no data).")
            continue
        df_to_insert.to_sql(
            name=table_name,
            con=conn,
            schema=SQL_SCHEMA,
            if_exists="append",
            index=False
        )
        print(f"  Inserted {len(df_to_insert)} row(s) → [{SQL_SCHEMA}].[{table_name}]")

print("\nDone! All metadata written to SQL Server.")

## 10. Quick verification queries

In [ ]:
for table_name in TABLE_MAP:
    result = pd.read_sql(
        f"SELECT TOP 5 * FROM [{SQL_SCHEMA}].[{table_name}] ORDER BY id DESC",
        engine
    )
    print(f"\n── {table_name} ──")
    display(result)

## 11. Useful SQL queries for downstream analysis

```sql
-- All PII-positive columns in a dataset
SELECT column_name, detected_entity_types, pct_flagged_rows, avg_confidence
FROM dbo.metadata_pii_column_summary
WHERE dataset_name = 'sample_employees'
  AND is_pii = 1
ORDER BY avg_confidence DESC;

-- High-confidence PII cells (>= 0.85)
SELECT column_name, row_index, cell_value, entity_type, confidence
FROM dbo.metadata_pii_cell_detail
WHERE dataset_name = 'sample_employees'
  AND confidence >= 0.85
ORDER BY confidence DESC;

-- Join column stats with PII summary for a full column inventory
SELECT cs.column_name, cs.column_type, cs.pct_missing,
       pii.is_pii, pii.detected_entity_types, pii.avg_confidence
FROM dbo.metadata_column_stats   cs
LEFT JOIN dbo.metadata_pii_column_summary pii
       ON cs.column_name  = pii.column_name
      AND cs.dataset_name = pii.dataset_name
      AND cs.run_timestamp= pii.run_timestamp
WHERE cs.dataset_name = 'sample_employees';
```